In [1]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed
)
from datasets import Dataset as ds
import random
from peft import PeftModel
import pandas as pd
from tqdm.auto import tqdm

# Configurar semillas para facilitar la reproducibilidad de los resultados
seed = 44
torch.manual_seed(seed)
random.seed(seed)
set_seed(seed)

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'mps'

In [2]:
base_prompt = """Using the next instructions, write a plain-language summary for patients:
* Organize it using short, reader-friendly headings similar to those used in patient-oriented evidence summaries (e.g., Review question, Background, Study characteristics, Key results, Conclusions, Quality of evidence).
* Do not use the technical headings verbatim; instead, adapt them into simple, descriptive labels.
* Integrate all information into a clear, coherent, and easy-to-follow narrative.
* Keep the language at or below a 6th-grade reading level.
* Avoid jargon; if you must use a technical term, explain it in simple, familiar words.
* Use active voice, mostly short words (one or two syllables), and sentences of no more than 20 words.
* Organize the summary into short paragraphs of 3-5 sentences
* Use simple numbers or ratios (for example, 1 in 2) instead of percentages.
* Do not add, remove, or infer any information not present in the abstract.
* The summary should remain consistent regardless of the order of sentences in the original abstract.
* Provide only the summary as plain text. Do not use bold, italics, headings, asterisks, or any other formatting
* Do not provide comments or explanations.
* Target a total length of 300–500 words.
Here is the abstract of a biomedical study to summarize:"""

In [3]:
def format_sample(sample, base_prompt):
    '''
    Función para crear prompt a partir de un texto técnico
    '''
    non_pls = sample

    prompt = (
        f"{base_prompt}\n\n"
        f"{non_pls}\n\n"
        "Plain-language summary:\n\n"
    )
    return prompt

In [1]:
version_modelo = '../modelos/Qwen3-1.7B-01'

In [5]:
model = AutoModelForCausalLM.from_pretrained(version_modelo,device_map='auto',torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(version_modelo, trust_remote_code=True)
tokenizer.add_special_tokens({'pad_token': '<|image_pad|>'})
model.eval()
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
def generate_pls(model, tokenizer, dataset, nombre, base_prompt=base_prompt):
    model.eval()
    results = []

    for sample in tqdm(dataset, total=len(dataset)):
        indice = sample['codigo_comun']
        non_pls = sample['non_pls']
        pls = sample['pls']
        prompt = format_sample(non_pls, base_prompt)

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                temperature=0.8,
                top_p=0.9,
                top_k=50,
                min_p=0,
                max_new_tokens=2000,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=[tokenizer.eos_token_id, tokenizer.pad_token_id]
            )

        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        _, _, pls_model = decoded.partition(prompt)
        results.append((indice, non_pls, pls, pls_model))

    nombre = nombre.partition('modelos/')[-1]
    nombre_col = f"pls_{nombre}"
    archivo = f"../datos/resumenes_generados/resultados_{nombre}.csv"

    pd.DataFrame(results, columns=['codigo_comun','non_pls','pls',nombre_col]).to_csv(archivo, sep=';', index=False)
    
    return None

Se carga el dataset de evaluación.

In [ ]:
df = pd.read_excel('../datos/Cochrane/test/test.xlsx')
ds_test = ds.from_pandas(df)

In [8]:
generate_pls(model, tokenizer, ds_test, version_modelo)

  0%|          | 0/218 [00:00<?, ?it/s]